# Automatic Music Transcription

In [ ]:
# install relevant packages for Google Colab
# !pip install pyguitarpro
# !pip install --upgrade attrs
# !pip install accelerate -U
# !pip install datasets

### Preprocessing

In [104]:
from sklearn.model_selection import train_test_split
import IPython.display as ipd
import copy
import torch
import torch.nn.functional as F
import librosa
import guitarpro
from guitarpro.models import Note, Beat
from transformers import AutoFeatureExtractor, WhisperModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'{device=}, {torch.__version__=}, {torch.cuda.is_available()=}, {torch.version.cuda=}')

device=device(type='cuda'), torch.__version__='2.1.2', torch.cuda.is_available()=True, torch.version.cuda='12.1'


In [82]:
song_filepath = 'dataset/john-mayer/30_Why_Georgia.mp3'
tab_filepath = 'dataset/john-mayer/3_why-georgia.gp5'
vocab_size = 129
start_token_idx = 128
sr = 16000

# Load in the song and tab
y, sr = librosa.load(song_filepath, sr=sr)
song = guitarpro.parse(tab_filepath)

print(f'{len(y)=}')

len(y)=484922


In [83]:
def tokenize_note_simple(note):
    """
    Tokenizes a note based on its string and fret number.
    """

    string = note.string
    value = note.value

    # Add pre-conditions for the note
    if string < 1 or string > 6:
        raise ValueError(f"Invalid string number: {string}")

    if value < 0 or value > 20:
        raise ValueError(f"Invalid fret number: {value}")

    # Calculate the token index based on string and fret
    # Token range for fretted notes: 1 to 120
    # Token range for open strings: 121 to 126
    if value == 0:  # Open string
        token_index = 121 + (6 - string)
    else:
        token_index = (string - 1) * 20 + value

    return token_index

In [84]:
g_note = Note(Beat(voice=None), string=6, value=3)
g_note_token_idx = tokenize_note_simple(g_note)

print(f'{g_note_token_idx=} for {g_note}')

g_note_token_idx=103 for Note(value=3, velocity=95, string=6, effect=<guitarpro.models.NoteEffect object at 0x7f39429b9890>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.rest: 0>)


In [85]:
def detokenize_note_simple(token_index):
    """
    Detokenizes a token index to find the corresponding guitar string and fret number
    """

    # Pre-conditions for the token index
    if token_index < 1 or token_index > vocab_size:
        raise ValueError(f"Invalid token index: {token_index}")

    # Determining the string and fret based on token index
    if 121 <= token_index <= 126:  # Open string
        string = 6 - (token_index - 121)
        fret = 0
    elif token_index == 127:
        return "wait"
    elif token_index == start_token_idx:
        return "start"
    elif token_index == 129:
        return "end"
    else:
        string = ((token_index - 1) // 20) + 1
        fret = token_index - ((string - 1) * 20)

    return f"s{string}f{fret}"

# Example usage
token_index = 126
note = detokenize_note_simple(token_index)
print(f'{token_index=} maps to {note=}')


token_index=126 maps to note='s1f0'


In [86]:
def tokenize_song_simple(song):
    """
    Tokenizes an entire song using a simplified scheme.
    """
    all_measures_tokens = []

    # Start token for the song
    all_measures_tokens.append([128])  # 128 is the start token

    for track in song.tracks:
        for measure in track.measures:
            measure_tokens = []
            for voice in measure.voices:
                for beat in voice.beats:
                    if track.isPercussionTrack:
                        continue  # Skip percussion tracks
                    else:
                        for note in beat.notes:
                            token = tokenize_note_simple(note)
                            measure_tokens.append(token)

                    # Assuming each beat is a 'wait' token
                    measure_tokens.append(127)  # 127 is the wait/rest token

            # Repeat measures if necessary
            if measure.repeatClose > 0:
                for _ in range(measure.repeatClose):
                    all_measures_tokens.append(measure_tokens)
            else:
                all_measures_tokens.append(measure_tokens)

    # End token for the song
    all_measures_tokens.append([129])  # 129 is the end token

    return all_measures_tokens

In [87]:
# tokenize the acoustic song track only
acoustic_song = copy.deepcopy(song)
acoustic_song.tracks = [acoustic_song.tracks[0]]

tokens = tokenize_song_simple(acoustic_song)

## Audio Segmentation
* Split the audio into segments of roughly the same length as the tablature measures.

In [91]:
# Given parameters
sample_rate = sr
chunk_duration = 2.5  # Chunk duration in seconds, this is the approximate length for each measure
hop_length = 512  # This is the default hop length for librosa's mfcc calculation

# Calculate the number of samples per chunk
samples_per_chunk = int(sample_rate * chunk_duration)

# Calculate the number of chunks
num_chunks = len(y) // samples_per_chunk + 1

# calculate total number of frames
total_frames = len(y) // hop_length + 1

# Calculate the number of frames per chunk
# Formula: Number of Frames = (Number of Samples in Segment - Frame Length) / Hop Length + 1
# Assuming the frame length is equal to the hop length for simplicity
frames_per_chunk = (samples_per_chunk) // hop_length + 1

print(f'{samples_per_chunk=}, {num_chunks=}, {frames_per_chunk=}, {total_frames=}')

samples_per_chunk=40000, num_chunks=13, frames_per_chunk=79, total_frames=948


In [106]:
# Initialize an empty list to hold the audio segments
audio_segments = []

# Iterate over the waveform and extract segments
for i in range(num_chunks):
    # Calculate the start and end index for each chunk
    start_idx = i * samples_per_chunk
    end_idx = start_idx + samples_per_chunk

    # Make sure the end index does not exceed the length of the waveform
    end_idx = min(end_idx, len(y))

    # Extract the segment and add it to the list
    segment = y[start_idx:end_idx]
    audio_segments.append(segment)

# Now, audio_segments contains the segmented audio


### Use a seq2seq transformer

In [109]:
model = WhisperModel.from_pretrained("openai/whisper-base")
feature_extractor = AutoFeatureExtractor.from_pretrained("openai/whisper-base")

In [110]:
mel_filter_bank = feature_extractor(y, sampling_rate=sr, return_tensors="pt")

In [111]:
my_input = mel_filter_bank.input_features
my_input.shape

torch.Size([1, 80, 3000])

In [112]:
decode_input_ids = torch.tensor([start_token_idx])
print(my_input.shape, decode_input_ids.shape)

torch.Size([1, 80, 3000]) torch.Size([1])


In [113]:
# Assuming model and device are already defined
model.to(device)

# Move input data to the same device as the model
my_input = my_input.to(device)
decoder_input_ids = decode_input_ids.to(device)

# Now you can pass the input data to the model
lhs = model(my_input, decoder_input_ids=decoder_input_ids)

ValueError: not enough values to unpack (expected 2, got 1)

In [83]:
lhs.last_hidden_state.shape

torch.Size([1, 31, 512])

In [84]:
vocab_size

129

To interpret the last_hidden_state tensor from your model and compare its predictions with your expected guitar tabs tokens, you need to convert the output from the model's hidden states to actual token predictions. This typically involves a few steps:

    Project the Hidden States to Token Logits: The last_hidden_state contains hidden representations, not direct token predictions. You need to project these representations onto the space of your possible tokens (e.g., your guitar tabs tokens). This is often done using a linear layer followed by a softmax layer in the model to convert hidden states into logits, representing probabilities for each token in your vocabulary.

    Convert Logits to Token IDs: Choose the token with the highest probability (logit) at each time step. This is typically done using the argmax operation.

    Map Token IDs to Actual Tokens: Convert the predicted token IDs back to the actual tokens (e.g., guitar tab notations) using the tokenizer or mapping you have.

    Compare with Expected Tokens: Compare these predicted tokens with your expected tokens (ground truth) to assess the model's performance.

Here is a general outline of how you might implement this:

In [89]:
import torch

# Assuming model, tokenizer, and lhs are already defined
# lhs.last_hidden_state is the output from your model

# Replace 512 with the size of your hidden states and vocab_size with the size of your vocabulary
linear_layer = torch.nn.Linear(512, vocab_size)

# Move the linear layer to the same device as your model and lhs.last_hidden_state
linear_layer = linear_layer.to(lhs.last_hidden_state.device)

# Now you can apply the linear layer to the last hidden state
logits = linear_layer(lhs.last_hidden_state)

torch.Size([1, 31, 129])

In [ ]:
# Convert logits to token IDs
predicted_token_ids = torch.argmax(logits, dim=-1)

In [88]:
predicted_token_ids.shape

torch.Size([1, 31])

In [91]:
# Map token IDs to actual tokens
for predicted_token_id in predicted_token_ids[0]:
  predicted_note = detokenize_note_simple(predicted_token_id)
  print(predicted_note)

# Now, predicted_tokens contains the predicted sequence
# Compare these with your expected tokens

s5f11
s5f11
s4f10
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20
s3f20


In [93]:
# alright we made a prediction.  what were the target tokens?
# and then how can we evaluate the performance?


AttributeError: 'list' object has no attribute 'shape'